# 🏐 Scot AI - Continue Roboflow Training on Colab / Camber GPU
### Resume & Fine-Tune Model: `yussef-essam/volleyball-actions-bg8ht-jctrk-2-yolo26x-t1`

This notebook allows you to **continue training your Roboflow volleyball model** without spending any Roboflow credits:
1. **Load Checkpoint**: Uses your pre-trained `weights.pt` (~118 MB, 14 volleyball action classes).
2. **Download Full Dataset**: Automatically pulls all 13,925 images from `yussef-essam/volleyball-actions-bg8ht-jctrk` (v2) for free via Roboflow API.
3. **Free Cloud GPU Training**: Fine-tunes with high-speed CUDA acceleration on Google Colab (T4/A100) or Camber Cloud.
4. **Export & Optional Deploy**: Saves the updated `best.pt` and can register it back to Roboflow if desired.

> **GPU Setup**: In Google Colab, make sure you select **Runtime** → **Change runtime type** → **T4 GPU**.

In [7]:
# [CELL 1] Check GPU Acceleration & Hardware Details
import importlib
import subprocess

has_torch = False
try:
    torch = importlib.import_module("torch")
    has_torch = True
except ImportError:
    pass

print("=" * 65)
if has_torch and torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[✓] GPU DETECTED : {gpu_name} ({vram:.1f} GB VRAM)")
    print("[✓] Ready for high-speed YOLO training!")
else:
    print("[!] NOTICE: Running on CPU runtime.")
    print("    TIP: In Colab, go to: Runtime -> Change runtime type -> Select T4 GPU")
print("=" * 65)

try:
    subprocess.run(["nvidia-smi"], check=False)
except Exception:
    pass

[!] NOTICE: Running on CPU runtime.
    TIP: In Colab, go to: Runtime -> Change runtime type -> Select T4 GPU


In [8]:
# [CELL 2] Install Core Computer Vision & Roboflow Dependencies
%pip install --quiet --upgrade ultralytics supervision roboflow opencv-python-headless matplotlib pyyaml
print("[✓] Dependencies installed successfully!")

[✓] Dependencies installed successfully!


In [9]:
# [CELL 3] Locate weights.pt (Sidebar Drag & Drop or Google Drive)
import importlib
import os
import shutil
from pathlib import Path

candidates = [
    Path('weights.pt'),
    Path('/content/weights.pt'),
    Path('/content/drive/MyDrive/weights.pt'),
    Path('/content/drive/MyDrive/Scot_AI/weights.pt'),
]

found_path = None
for p in candidates:
    if p.exists() and p.is_file():
        found_path = p
        break

# If found in Drive, copy to local /content for fast GPU access
if found_path and str(found_path) != 'weights.pt' and str(found_path) != '/content/weights.pt':
    print(f'[*] Copying {found_path} to working directory...')
    shutil.copy(str(found_path), 'weights.pt')
    found_path = Path('weights.pt')

if found_path and found_path.exists():
    size_mb = found_path.stat().st_size / (1024**2)
    print('=' * 65)
    print(f'[✓] CHECKPOINT READY: {found_path.resolve()} ({size_mb:.1f} MB)')
    ultralytics = importlib.import_module('ultralytics')
    YOLO = getattr(ultralytics, 'YOLO')
    m = YOLO(str(found_path))
    print(f'[✓] Detected {len(m.names)} classes:')
    for cid, cname in m.names.items():
        print(f'    [{cid:2d}] {cname}')
    print('=' * 65)
else:
    print('=' * 65)
    print("[!] 'weights.pt' IS NOT IN /content YET.")
    print('=' * 65)
    print('👉 FASTEST FIX (Drag & Drop in 5 seconds):')
    print('   1. In Google Colab, click the 📁 Folder icon on the LEFT sidebar.')
    print('   2. Drag weights.pt from your computer into that folder pane.')
    print('   3. Wait for the upload circle at the bottom-left to finish.')
    print('   4. Re-run this cell!')
    print('-' * 65)
    print('👉 OR VIA GOOGLE DRIVE (If you uploaded it to Drive):')
    print('   from google.colab import drive; drive.mount("/content/drive")')
    print('   !cp /content/drive/MyDrive/weights.pt ./weights.pt')
    print('=' * 65)


[!] 'weights.pt' IS NOT IN /content YET.
👉 FASTEST FIX (Drag & Drop in 5 seconds):
   1. In Google Colab, click the 📁 Folder icon on the LEFT sidebar.
   2. Drag weights.pt from your computer into that folder pane.
   3. Wait for the upload circle at the bottom-left to finish.
   4. Re-run this cell!
-----------------------------------------------------------------
👉 OR VIA GOOGLE DRIVE (If you uploaded it to Drive):
   from google.colab import drive; drive.mount("/content/drive")
   !cp /content/drive/MyDrive/weights.pt ./weights.pt


In [10]:
# [CELL 4] Download Roboflow Dataset: 'yussef-essam/volleyball-actions-bg8ht-jctrk' (v2)
# Note: Downloading datasets is 100% FREE and uses 0 Roboflow credits!
import importlib
from pathlib import Path

roboflow_mod = importlib.import_module("roboflow")
Roboflow = getattr(roboflow_mod, "Roboflow")

ROBOFLOW_API_KEY = "sbFeSa7pqMcxrLUEZY67"
WORKSPACE_SLUG = "yussef-essam"
PROJECT_SLUG = "volleyball-actions-bg8ht-jctrk"
VERSION_NUM = 2
DATASET_DIR = Path("./dataset/volleyball_actions_v2").resolve()

print(f"[*] Connecting to Roboflow workspace '{WORKSPACE_SLUG}'...")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_SLUG).project(PROJECT_SLUG)
version = project.version(VERSION_NUM)

print(f"[*] Downloading version {VERSION_NUM} in YOLOv11 format...")
dataset = version.download("yolov11", location=str(DATASET_DIR))
print(f"[✓] Dataset successfully loaded at: {dataset.location}")

data_yaml_path = Path(dataset.location) / "data.yaml"
if data_yaml_path.exists():
    print("\n--- data.yaml preview ---")
    print(data_yaml_path.read_text().strip())
    print("-------------------------")

[*] Connecting to Roboflow workspace 'yussef-essam'...
loading Roboflow workspace...
loading Roboflow project...
[*] Downloading version 2 in YOLOv11 format...
Exporting format yolov11 in progress : 95.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to /content/dataset/volleyball_actions_v2 in yolov11:: 100%|██████████| 27854/27854 [00:17<00:00, 1561.09it/s]


[✓] Dataset successfully loaded at: /content/dataset/volleyball_actions_v2

--- data.yaml preview ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 14
names: ['a_surve', 'attacker', 'b_surve', 'block', 'blocker', 'blocking', 'defense', 'digging', 'over', 'passing', 'serving', 'setting', 'spike', 'spiking']

roboflow:
  workspace: yussef-essam
  project: volleyball-actions-bg8ht-jctrk
  version: 2
  license: CC BY 4.0
  url: https://app.roboflow.com/yussef-essam/volleyball-actions-bg8ht-jctrk/2
-------------------------


In [11]:
# [CELL 5] Continue / Fine-Tune Training from 'weights.pt'
import importlib
from pathlib import Path

torch = importlib.import_module("torch")
ultralytics = importlib.import_module("ultralytics")
YOLO = getattr(ultralytics, "YOLO")

device = 0 if torch.cuda.is_available() else "cpu"
data_yaml = str(Path("./dataset/volleyball_actions_v2/data.yaml").resolve())

print(f"[*] Initializing YOLO fine-tuning from 'weights.pt' on device: {device}...")
model = YOLO("weights.pt")

# Hyperparameters tuned for continuation:
# lr0=0.001 : gentle learning rate to prevent catastrophic forgetting of existing weights
# mosaic=1.0: keeps fast small volleyball balls visible
# cos_lr=True: smooth cosine annealing schedule
results = model.train(
    data=data_yaml,
    epochs=80,
    batch=16 if device == "cpu" else 32,
    imgsz=640,
    patience=20,
    lr0=0.001,
    cos_lr=True,
    mosaic=1.0,
    device=device,
    project="./runs/scot_train",
    name="volleyball_continued_roboflow",
    save=True,
    save_period=10,
    plots=True,
)

print("[✓] Continued training complete!")
best_weights = Path("./runs/scot_train/volleyball_continued_roboflow/weights/best.pt")
print(f"    Best weights saved at: {best_weights.resolve()}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
[*] Initializing YOLO fine-tuning from 'weights.pt' on device: cpu...


FileNotFoundError: [Errno 2] No such file or directory: 'weights.pt'

In [ ]:
# [CELL 6] Evaluate Validation & Test Split Metrics
import importlib
from pathlib import Path

ultralytics = importlib.import_module("ultralytics")
YOLO = getattr(ultralytics, "YOLO")

best_weights = Path("./runs/scot_train/volleyball_continued_roboflow/weights/best.pt")
active_weights = str(best_weights) if best_weights.exists() else "weights.pt"
eval_model = YOLO(active_weights)

data_yaml = str(Path("./dataset/volleyball_actions_v2/data.yaml").resolve())
val_metrics = eval_model.val(data=data_yaml, split="test")

print("=" * 65)
print("FINAL TEST METRICS (14 Volleyball Action Classes):")
print(f"  mAP @ 0.50   : {val_metrics.box.map50:.4f} ({val_metrics.box.map50 * 100:.1f}%)")
print(f"  mAP @ 50-95  : {val_metrics.box.map:.4f} ({val_metrics.box.map * 100:.1f}%)")
print(f"  Precision    : {val_metrics.box.mp:.4f} ({val_metrics.box.mp * 100:.1f}%)")
print(f"  Recall       : {val_metrics.box.mr:.4f} ({val_metrics.box.mr * 100:.1f}%)")
print("=" * 65)

In [ ]:
# [CELL 7] Visual Inference & Prediction on Test Frames
import importlib
from pathlib import Path

cv2 = importlib.import_module("cv2")
plt = importlib.import_module("matplotlib.pyplot")
ultralytics = importlib.import_module("ultralytics")
YOLO = getattr(ultralytics, "YOLO")

best_weights = Path("./runs/scot_train/volleyball_continued_roboflow/weights/best.pt")
model = YOLO(str(best_weights) if best_weights.exists() else "weights.pt")

test_images = list(Path("./dataset/volleyball_actions_v2/test/images").glob("*.jpg"))
if test_images:
    sample_img = str(test_images[0])
    results = model(sample_img)
    annotated_frame = results[0].plot()

    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
    plt.title(f"Scot AI Volleyball Detections - {Path(sample_img).name}")
    plt.axis("off")
    plt.show()
    print("[✓] Visual inference rendered successfully!")
else:
    print("[!] No test images found.")

In [ ]:
# [CELL 8] Download Checkpoint / (Optional) Deploy to Roboflow
import importlib
from pathlib import Path

best_weights = Path("./runs/scot_train/volleyball_continued_roboflow/weights/best.pt")

# 1. Download to local machine via browser if in Colab
try:
    google_colab = importlib.import_module("google.colab")
    files = getattr(google_colab, "files")
    if best_weights.exists():
        print(f"[*] Triggering browser download for {best_weights.name}...")
        files.download(str(best_weights))
except Exception:
    pass

# 2. Optional: Deploy updated weights back to Roboflow version
# Uncomment below if you want Roboflow to host and serve this model endpoint:
#
# roboflow_mod = importlib.import_module("roboflow")
# Roboflow = getattr(roboflow_mod, "Roboflow")
# rf = Roboflow(api_key="sbFeSa7pqMcxrLUEZY67")
# version = rf.workspace("yussef-essam").project("volleyball-actions-bg8ht-jctrk").version(2)
# version.deploy(
#     model_type="yolov8x",
#     model_path=str(best_weights.parent.parent),
#     filename="weights/best.pt"
# )
print("[✓] All steps completed!")